In [1]:
# IMPORTS
import numpy as np
import pandas as pd
import requests


#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt
from io import StringIO

### Question 1: What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?

In [15]:
url = f"https://www.iposcoop.com/ipos-recently-filed/"
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/58.0.3029.110 Safari/537.3'
    )
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

# Wrap HTML text in StringIO to avoid deprecation warning
# "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
html_io = StringIO(response.text)
tables = pd.read_html(html_io)
tables = pd.DataFrame(tables[0])
tables = tables[tables['Expected To Trade'] == 'Withdrawn']
tables = tables[tables['File Date'] < '2026-09-11']
tables

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O
50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,$22.50,Withdrawn,S/O
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,$200.00,Withdrawn,S/O
69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,$10.00,$10.00,$125.00,Withdrawn,S/O
84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,$4.00,$6.00,$19.00,Withdrawn,S/O
128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,$50.40,Withdrawn,S/O
158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,$4.00,$4.00,$15.00,Withdrawn,S/O
163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,$15.94,Withdrawn,S/O


In [17]:
import re

rules = [
    (r"\bTechnologies\b", "Technologies"),
    (r"\bAcquisition\s+Corp(?:oration)?\b|\bCorp\b", "Acquisition Corp"),
    (r"\bInc(?:orporated)?\b", "Inc."),
    (r"\bGroup\b", "Group"),
    (r"\bLtd\b|\bLimited\b", "Limited"),
    (r"\bHoldings?\b", "Holdings"),
]

for idx, name in tables['Company'].items():
    match_found = False
    for pattern, label in rules:
        if re.search(pattern, str(name), flags=re.IGNORECASE):
            tables.at[idx, 'Company type'] = label
            match_found = True
            break          # stop at first match
    if not match_found:
        tables.at[idx, 'Company type'] = "Other"

In [32]:
tables['Price High'] = pd.to_numeric(tables['Price High'].astype(str).str.replace(r"[^\d.]", "", regex=True), errors='coerce')
tables['Price Low'] = pd.to_numeric(tables['Price Low'].astype(str).str.replace(r"[^\d.]", "", regex=True), errors='coerce')
tables['Avg_price'] = (tables['Price High'] + tables['Price Low'])/2
tables['Shares (millions)'] = pd.to_numeric(tables['Shares (millions)'].astype(str).str.replace(r"[^\d.]", "", regex=True), errors='coerce')
tables['Est $ Vol (millions)'] = pd.to_numeric(tables['Est $ Vol (millions)'].astype(str).str.replace(r"[^\d.]", "", regex=True), errors='coerce')
tables['Shares_offered_value'] = np.where((tables['Shares (millions)'] * tables['Avg_price']).notna(), tables['Shares (millions)'] * tables['Avg_price'], tables['Est $ Vol (millions)'])
tables.groupby(['Company type']).sum()['Shares_offered_value']

,Shares_offered_value
Company type,
Acquisition Corp,499.9850
Group,32.5000
Holdings,311.6575
Inc.,351.0000
Limited,197.2500
Other,290.4450
Technologies,184.9000


### Question 2: What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?

In [24]:
url = f"https://www.iposcoop.com/2025-pricings/"
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/58.0.3029.110 Safari/537.3'
    )
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

# Wrap HTML text in StringIO to avoid deprecation warning
# "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
html_io = StringIO(response.text)
df2 = pd.read_html(html_io)
df2 = pd.DataFrame(df2[0])

# Convert 'Offer Date' to datetime objects for proper comparison
df2['Offer Date'] = pd.to_datetime(df2['Offer Date'], format='%m/%d/%Y')
comparison_date = pd.to_datetime('09/01/2025', format='%m/%d/%Y')


df2 = df2[(df2['Offer Date'] < comparison_date) & (df2['Return'] != '0.00%')]
df2

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00%,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00%,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75%,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-90.75%,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,1.20%,S/O
...,...,...,...,...,...,...,...,...,...,...
226,Mint Inc. Ltd.,MIMI,Other,2025-01-10,1.8,$4.00,$4.11,$5.30,32.50%,S/O
227,3 E Network Technology Group Limited,MASK,Technology,2025-01-08,1.3,$4.00,$3.25,$2.75,-31.25%,S/O
228,"Cantor Equity Partners I, Inc. (Stock-Only SPAC)",CEPO,Blank Check,2025-01-07,20.0,$10.00,$10.05,$10.10,1.00%,S/O
229,Zhengye Biotechnology Holding Limited,ZYBT,Health Care,2025-01-07,1.5,$4.00,$4.86,$4.22,5.50%,S/O


In [25]:
stocks_data = []
for ticker in df2['Symbol']:
  ticker_obj = yf.Ticker(ticker)
  ticker_df = ticker_obj.history(period = "max")

  if ticker_df.empty:
    continue

  else:
    ticker_df["Ticker"] = ticker
    stocks_data.append(ticker_df)

stocks_df = pd.concat(stocks_data)
stocks_df = stocks_df.sort_values(['Ticker', 'Date'])

ERROR:yfinance:$MJID: possibly delisted; no timezone found
ERROR:yfinance:$EMPG: possibly delisted; no timezone found
ERROR:yfinance:$CAEP: possibly delisted; no timezone found
ERROR:yfinance:$PTNM: possibly delisted; no timezone found
ERROR:yfinance:$AHL: possibly delisted; no timezone found
ERROR:yfinance:$CEPT: possibly delisted; no timezone found
ERROR:yfinance:$SDM: possibly delisted; no timezone found
ERROR:yfinance:$TBH: possibly delisted; no timezone found
ERROR:yfinance:$WGRX: possibly delisted; no timezone found
ERROR:yfinance:$AGH: possibly delisted; no timezone found
ERROR:yfinance:$EPWK: possibly delisted; no timezone found
ERROR:yfinance:$MTSR: possibly delisted; no timezone found
ERROR:yfinance:$SKBL: possibly delisted; no timezone found
ERROR:yfinance:$MCTR: possibly delisted; no timezone found


In [27]:
stocks_df['growth_252d'] = (
    stocks_df.groupby('Ticker')['Close']
    .transform(lambda x: x / x.shift(252))
)

stocks_df['volatility'] = (
    stocks_df.groupby('Ticker')['Close']
    .transform(lambda x: x.rolling(30).std() * np.sqrt(252))
)

stocks_df['Sharpe'] = (
    (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']
)

stocks_df = stocks_df.reset_index()
stocks_df = stocks_df[stocks_df['Date'].dt.date == pd.to_datetime('2026-09-11').date()]
median_sharpe_ratio = stocks_df['Sharpe'].median()
print(f"Median Sharpe ratio: {median_sharpe_ratio}")

Median Sharpe ratio: 0.050142446812154115


### Question 3: What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?

In [31]:
stocks_data = []
for ticker in df2['Symbol']:
  ticker_obj = yf.Ticker(ticker)
  ticker_df = ticker_obj.history(period = "max")

  if ticker_df.empty:
    continue

  else:
    ticker_df["Ticker"] = ticker
    stocks_data.append(ticker_df)

stocks = pd.concat(stocks_data)
stocks = stocks.sort_values(['Ticker', 'Date'])
stocks

ERROR:yfinance:$MJID: possibly delisted; no timezone found
ERROR:yfinance:$EMPG: possibly delisted; no timezone found
ERROR:yfinance:$CAEP: possibly delisted; no timezone found
ERROR:yfinance:$PTNM: possibly delisted; no timezone found
ERROR:yfinance:$AHL: possibly delisted; no timezone found
ERROR:yfinance:$CEPT: possibly delisted; no timezone found
ERROR:yfinance:$SDM: possibly delisted; no timezone found
ERROR:yfinance:$TBH: possibly delisted; no timezone found
ERROR:yfinance:$WGRX: possibly delisted; no timezone found
ERROR:yfinance:$AGH: possibly delisted; no timezone found
ERROR:yfinance:$EPWK: possibly delisted; no timezone found
ERROR:yfinance:$MTSR: possibly delisted; no timezone found
ERROR:yfinance:$SKBL: possibly delisted; no timezone found
ERROR:yfinance:$MCTR: possibly delisted; no timezone found


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,Capital Gains
Date,,,,,,,,,
2025-01-24 00:00:00-05:00,17.250000,18.000000,16.500000,17.379999,2217300,0.0,0.0,AAPG,NaN
2025-01-27 00:00:00-05:00,17.389999,17.500000,17.000000,17.250000,315800,0.0,0.0,AAPG,NaN
2025-01-28 00:00:00-05:00,17.370001,17.379999,17.025000,17.200001,39800,0.0,0.0,AAPG,NaN
2025-01-29 00:00:00-05:00,17.389999,17.450001,17.190001,17.330000,121000,0.0,0.0,AAPG,NaN
2025-01-30 00:00:00-05:00,17.450001,17.860001,17.280001,17.600000,228900,0.0,0.0,AAPG,NaN
...,...,...,...,...,...,...,...,...,...
2026-09-22 00:00:00-04:00,1.350000,1.350000,1.260000,1.280000,24400,0.0,0.0,ZYBT,NaN
2026-09-23 00:00:00-04:00,1.280000,1.380000,1.280000,1.370000,28300,0.0,0.0,ZYBT,NaN
2026-09-24 00:00:00-04:00,1.301000,1.370000,1.300000,1.310000,8800,0.0,0.0,ZYBT,NaN


In [43]:
for month in range(1, 13):
    days = month * 21

    stocks[f'future_growth_{month}_m'] = (
        stocks.groupby('Ticker')['Close']
        .transform(lambda x: x.shift(-days) / x)
    )

In [45]:
stocks.reset_index(inplace=True)

In [46]:
min_dates = (
    stocks.groupby('Ticker')['Date']
    .min()
    .reset_index()
)

min_dates.columns = ['Ticker', 'min_date']

In [47]:
entry_data = stocks.merge(
    min_dates,
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'min_date'],
    how='inner'
)

In [48]:
growth_columns = [
    f'future_growth_{month}_m'
    for month in range(1, 13)
]

stats = entry_data[growth_columns].describe()

print(stats)

       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         132.000000         131.000000         131.000000   
mean            1.119635           1.294861           1.252014   
std             1.254616           1.957999           1.912626   
min             0.072941           0.106197           0.090147   
25%             0.708835           0.584863           0.442477   
50%             0.935351           0.890411           0.827160   
75%             1.098038           1.152849           1.149994   
max             9.466019          16.066783          16.344464   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         131.000000         131.000000         131.000000   
mean            1.135939           1.084528           1.114789   
std             1.558464           1.468733           1.668273   
min             0.063927           0.032245           0.022075   
25%             0.392759           0.400704           0.308026   
50%      

In [49]:
median_values = stats.loc['50%']

print(median_values)

future_growth_1_m     0.935351
future_growth_2_m     0.890411
future_growth_3_m     0.827160
future_growth_4_m     0.741678
future_growth_5_m     0.667857
future_growth_6_m     0.726000
future_growth_7_m     0.662000
future_growth_8_m     0.603508
future_growth_9_m     0.580294
future_growth_10_m    0.533333
future_growth_11_m    0.492453
future_growth_12_m    0.551321
Name: 50%, dtype: float64


In [50]:
highest_median_column = median_values.idxmax()
highest_median_value = median_values.max()

print(highest_median_column)
print(highest_median_value)

future_growth_1_m
0.9353509333873746
